In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io
import skimage.measure
from IPython.display import display, Markdown
import os

In [2]:
# Introduction
display(Markdown("""
# 📌 Coordinate Transformation Notebook

This notebook helps users transform coordinates from a global reference frame to a new coordinate system, generated using a plugin.
We will cover two use cases:

1. **Transforming manually provided points** into the new reference frame.
2. **Transforming cell centroids extracted from a segmentation mask** into the new reference frame.

Each section will guide you through the process, from loading data to saving transformed coordinates.
"""))


# 📌 Coordinate Transformation Notebook

This notebook helps users transform coordinates from a global reference frame to a new coordinate system, generated using a plugin.
We will cover two use cases:

1. **Transforming manually provided points** into the new reference frame.
2. **Transforming cell centroids extracted from a segmentation mask** into the new reference frame.

Each section will guide you through the process, from loading data to saving transformed coordinates.


In [2]:
# Function to load the coordinate system from CSV
def load_coordinate_system(csv_path):
    """
    Loads the new coordinate system's origin and basis vectors from a CSV file.
    The CSV should have three rows: the first for the origin, and the next two for the basis vectors.
    """
    df = pd.read_csv(csv_path, header=None)
    origin = list(map(float, df[0][1].strip("[]").split()))
    u, v, w = list(map(float, df[1][1].strip("[]").split())), list(map(float, df[2][1].strip("[]").split())),                       list(map(float, df[3][1].strip("[]").split()))

    return origin, u, v, w

# Function to transform points to the new coordinate system
def transform_points(points, origin, u, v, w):
    """
    Transforms a set of points into the new coordinate system.
    """
    transformation_matrix = np.column_stack((u, v, w))
    transformed_points = np.linalg.solve(transformation_matrix, (points - origin).T).T
    return transformed_points


In [35]:
# Load the CSV with coordinate system information
csv_path = "coordinate_system.csv"  # Replace with the actual file path
# Replace with the actual file path
origin, u, v, w = load_coordinate_system(csv_path)

In [36]:

# Section 1: Transforming user-provided points
display(Markdown("""
## 🟢 Section 1: Transforming User-Provided Points

If you have specific points whose coordinates you want to express in the new reference frame, you can input them below.
The transformed coordinates will be saved as a new CSV file.
"""))


## 🟢 Section 1: Transforming User-Provided Points

If you have specific points whose coordinates you want to express in the new reference frame, you can input them below.
The transformed coordinates will be saved as a new CSV file.


In [56]:

# Define the points (user must manually input their points)
points = np.array([
    [5.0, 10.0, 15.0],
    [8.0, 12.0, 20.0],
    [10.0, 15.0, 25.0]
])  # Replace with user points

# Transform the points
transformed_points = transform_points(points, origin, u, v, w)

# Save the transformed points
output_csv = "transformed_points.csv"
pd.DataFrame(transformed_points, columns=["X_new", "Y_new", "Z_new"]).to_csv(output_csv, index=False)
display(Markdown(f"✅ **Transformed points saved to `{output_csv}`**"))

✅ **Transformed points saved to `transformed_points.csv`**

In [10]:

display(Markdown("""
## 🔵 Section 2: Transforming Cell Centroids from a Segmentation Mask

Once an orthonormal reference frame has been created for the images in the folder,
this section allows us to compute the transformed coordinates of cell centroids
for each image that has an associated segmentation mask (the files must be named as follow : image_name_mask.tif).

### How it works:
1. **Extract centroids**: Identify the centroids of segmented regions using `regionprops`.
2. **Apply transformation**: Convert these centroids into the new reference frame.
3. **Save the results**: Store the transformed coordinates in a structured output CSV for each image.

By doing this, we ensure that the spatial information of segmented cells is consistently
aligned with the reference coordinate system, facilitating further quantitative analysis.
"""))



## 🔵 Section 2: Transforming Cell Centroids from a Segmentation Mask

Once an orthonormal reference frame has been created for the images in the folder,
this section allows us to compute the transformed coordinates of cell centroids
for each image that has an associated segmentation mask.

### How it works:
1. **Extract centroids**: Identify the centroids of segmented regions using `regionprops`.
2. **Apply transformation**: Convert these centroids into the new reference frame.
3. **Save the results**: Store the transformed coordinates in a structured output CSV for each image.

By doing this, we ensure that the spatial information of segmented cells is consistently
aligned with the reference coordinate system, facilitating further quantitative analysis.


In [11]:
def process_segmentation_folder(input_folder):
    """
    Process all segmentation masks in the input folder and transform the centroids.
    - Detects corresponding reference CSV files.
    - Saves the transformed centroids in a structured folder.

    Parameters:
    - input_folder: Path to the dataset containing images, masks, and reference CSVs.
    """

    # Get all segmentation masks in the folder
    mask_files = [f for f in os.listdir(input_folder) if f.endswith("_mask.tif")]

    for mask_file in mask_files:
        base_name = mask_file.replace("_mask.tif", "")
        mask_path = os.path.join(input_folder, mask_file)
        ref_csv_path = os.path.join(input_folder, base_name + ".csv")

        # Ensure the reference CSV exists
        if not os.path.exists(ref_csv_path):
            print(f"⚠️ Missing reference file for {base_name}, skipping.")
            continue

        # Load segmentation mask
        mask = skimage.io.imread(mask_path)

        # Extract cell centroids using regionprops
        props = skimage.measure.regionprops(mask)
        centroids = np.array([prop.centroid for prop in props])

        # Load the reference coordinate system
        origin, u, v, w = load_coordinate_system(ref_csv_path)

        # Transform centroids
        transformed_centroids = transform_points(centroids, origin, u, v, w)

        # Save the transformed centroids
        output_csv_path = os.path.join(input_folder, base_name + "_cells_coordinates.csv")
        pd.DataFrame(transformed_centroids, columns=["X_new", "Y_new", "Z_new"]).to_csv(output_csv_path, index=False)

        print(f"✅ Transformed centroids saved: {output_csv_path}")

In [13]:
# Define input and output directories
input_folder = "path/to/your/dataset"  # Change this to the actual dataset path

# Run the processing function
process_segmentation_folder(input_folder)

FileNotFoundError: [Errno 2] No such file or directory: 'path/to/your/dataset'